In [1]:
import os

This files explains the basic of langchain.
Type of messages
how to cretae a agent using langchain
how to get structured output
how to stream the output
how to bind tools to the llm

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
openai_api_key = os.getenv("OPENAI_API_KEY")

In [4]:
from langchain.agents import create_agent

In [5]:
def get_weather(location: str) -> str:
    """
    Function to get the weather for a given location.
    
    """
    # Placeholder implementation
    return f"The weather in {location} is sunny with a high of 25°C."

In [6]:
agent = create_agent(
    model = "gpt-5-nano",
    tools = [get_weather],
    system_prompt = "You are a helpful assistant that provides weather information.",
)

In [19]:
result = agent.invoke(
    {"messages":[{"role": "user", "content": "What's the weather in New York?"}]}
)

In [20]:
result

{'messages': [HumanMessage(content="What's the weather in New York?", additional_kwargs={}, response_metadata={}, id='ec6e0836-e3b0-4dbd-ac7f-cda0035f435c'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 88, 'prompt_tokens': 148, 'total_tokens': 236, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 64, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EDuocqmBCaJCbpME2g8coNTmTMnsU', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a010a0-cfea-79f3-b726-4b88d3a75ae2-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'New York'}, 'id': 'call_gN0onWqaLh4OlXLKjksR2saE

In [21]:
result["messages"][-1].content

'Today in New York: sunny with a high of 25°C (77°F).\n\nWould you like the current temperature, hourly forecast, or details for another location?'

In [7]:
from langchain.chat_models import init_chat_model

In [8]:
model = init_chat_model("gpt-5-nano",temperature=0)

In [9]:
agent1 = create_agent(
    model = model,
    tools = [get_weather],
    system_prompt = "You are a helpful assistant that provides weather information, based on the tool available. don't ask user anyother question just reply with tool output",
)

In [10]:
result = agent1.invoke(
    {"messages":[{"role": "user", "content": "What's the weather in New York?"}]}
)

In [11]:
result["messages"][-1].content

'The weather in New York is sunny with a high of 25°C.'

We can directly call the init model as well instead of passing it agent. then why we to pass model to agent. this model will act as a brain to the agent

In [12]:
model.invoke("what is the capital of France?")

AIMessage(content='Paris.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 139, 'prompt_tokens': 13, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EEpLZtfJ24ZMAUDN3P0XI4yHX35oh', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a01d94-a467-7e71-9a44-091fc0897ab2-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 13, 'output_tokens': 139, 'total_tokens': 152, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})

we can have streaming output as well

In [16]:
chunks =[]
full_message = None
for chunk in model.stream("write few sentence about langchain?"):
    chunks.append(chunk)
    full_message = chunk if full_message is None else full_message + chunk

print("Reconstructed full message:", full_message.content)

Reconstructed full message: LangChain is a framework for building applications powered by large language models (LLMs). It provides modular building blocks—prompt templates, chains to link steps, and agents that decide which tools to call and when. It also offers memory, vector stores, and integrations with tools and data sources to build multi-step, stateful workflows. Available for Python and TypeScript, LangChain is commonly used to create chatbots, question-answer systems, and automations that combine LLM reasoning with external actions.


In [18]:
chunks = []
full_message = None
for chunk in model.stream("write few sentence about langchain?"):
    chunks.append(chunk)
    full_message = chunk if full_message is None else full_message + chunk
    print(chunk.content, end="", flush=True)  # print as it streams



LangChain is an open-source framework for building AI-powered applications with large language models. It provides abstractions like Prompts, Chains, and Agents to compose LLM calls, handle tool usage, and manage memory. You can build chatbots, document Q&A, or automated workflows that retrieve data, run external tools, and perform computations. It supports Python and JavaScript/TypeScript and works with major LLM providers. In short, LangChain helps you structure multi-step reasoning and integrations when building LLM-driven apps.

In [20]:
from langchain_core.messages import HumanMessage, SystemMessage

In [22]:
messages = [
    SystemMessage(content="You are a helpful assistant, answers the user question poetically."),
    HumanMessage(content="What is lang chain?")
]

In [24]:
model = init_chat_model("gpt-5-nano",temperature=0.7)
result =model.invoke(messages)
print(result.content)

LangChain is not a language to learn, but a chain you weave,
A scaffold where language models are taught to act, to think, to weave.

In Python or TypeScript its threads are spun,
Prompts become templates, ready-made for action.

Chains are recipes, steps in a patient line,
Ask, retrieve, summarize, decide—one after the other, in time.

Agents are captains with a catalog of doors,
Tools and APIs, calculators, searchers, and more.

Memory keeps a diary of what came before,
So the model recalls the past when you open the door.

Vector stores, retrievers, documents in tow,
They ground the model’s wandering mind with glow.

Document loaders feed it books, PDFs, and streams,
Data becomes wisdom powering its schemes.

Tools are the keys that unlock the outside world,
LangChain choreographs how the model uses each unfurled.

Together they turn prompts from merely “what if” to “what do,”
A framework for building apps where ideas come true.

So LangChain is the loom and the map and the crew,
A 

simple way to add tools to model

In [40]:
model1 = init_chat_model("gpt-5-nano",temperature=0,max_tokens=3000)

In [41]:
def get_weather(location: str) -> str:
    """
    Function to get the weather for a given location.
    
    """
    # Placeholder implementation
    return f"The weather in {location} is sunny with a high of 25°C."

In [27]:
def set_password(password: str) -> str:
    """
    Function to set a password.
    
    """
    # Placeholder implementation
    return f"Password '{password}' has been set successfully."

In [28]:
model_with_tools = model.bind_tools([get_weather, set_password])

In [45]:
response =model_with_tools.invoke("what is the weather in New York?")

In [46]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'New York'},
  'id': 'call_wbFcXSka2sBh06oVZ6eyssI1',
  'type': 'tool_call'}]

In [49]:
for ans in response.tool_calls:
    print("function name:", ans["name"])
    print("arguments:", ans["args"])
    print("type:", ans["type"])

function name: get_weather
arguments: {'location': 'New York'}
type: tool_call


Structured output

In [32]:
from pydantic import BaseModel, Field

class EmailSchema(BaseModel):
    subject: str = Field(..., description="The subject of the email")
    body: str = Field(..., description="The body of the email")


In [42]:
model_with_structure = model1.with_structured_output(EmailSchema)

In [43]:
model_with_structure.invoke("write a email to my boss for leave")

EmailSchema(subject='Leave Request: [start date] to [end date]', body="Dear [Boss's Name],\n\nI hope you're well. I would like to request leave from [start date] to [end date] due to [brief reason]. I will ensure all my current tasks are up to date before I go, and I can hand over responsibilities to [colleague's name] during my absence. I will be reachable at [phone/email] for any urgent matters.\n\nThank you for considering my request. I appreciate your support.\n\nBest regards,\n[Your Name]")